# Nutrient Cycling in PyPath

## Overview
The nutrient cycling system models nutrient dynamics (N, P, Si, Fe) through producers, consumers, and recycling. This notebook demonstrates nutrient content, excretion, uptake, and stoichiometry.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pypath.core.params import (
    NutrientElement, NutrientContent, NutrientExcretion, 
    NutrientUptake, NutrientCycle
)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Part 1: Nutrient Elements

Define available nutrient types:

In [ ]:
# Show available nutrients
print('Nutrient Elements:')
for elem in NutrientElement:
    print(f'  {elem.name}: {elem.value}')

## Part 2: Nutrient Content

Define nutrient composition of biomass:

In [ ]:
# Create nutrient content for fish (per unit biomass)
cod_nutrients = NutrientContent(
    group_name='Cod',
    nitrogen_percent=2.5,      # 2.5% of wet weight
    phosphorus_percent=0.4,    # 0.4% of wet weight
    silicon_percent=0.01,      # minimal
    iron_mg_kg=10.0             # 10 mg/kg
)

phytoplankton_nutrients = NutrientContent(
    group_name='Phytoplankton',
    nitrogen_percent=5.0,      # high N content
    phosphorus_percent=0.8,
    silicon_percent=1.0,       # diatoms contain Si
    iron_mg_kg=50.0
)

print('Nutrient content (% dry weight):')
print(f'  Cod: N={cod_nutrients.nitrogen_percent}%, P={cod_nutrients.phosphorus_percent}%')
print(f'  Phytoplankton: N={phytoplankton_nutrients.nitrogen_percent}%, Si={phytoplankton_nutrients.silicon_percent}%')

## Part 3: Nutrient Excretion

Model nutrient loss from respiration and metabolism:

In [ ]:
# Define excretion rates (proportion of ingestion)
cod_excretion = NutrientExcretion(
    group_name='Cod',
    nitrogen_excretion_rate=0.08,  # 8% of ingestion
    phosphorus_excretion_rate=0.02,  # 2% of ingestion
    urine_proportion=0.6,  # 60% via urine, 40% via feces
    ammonium_fraction=0.8   # 80% as ammonium, rest organic
)

print('Excretion pathways (Cod):')
print(f'  N excretion rate: {cod_excretion.nitrogen_excretion_rate:.1%} of ingestion')
print(f'  Via urine: {cod_excretion.urine_proportion:.0%}')
print(f'  As ammonium: {cod_excretion.ammonium_fraction:.0%}')

## Part 4: Nutrient Uptake and Monod Kinetics

Model nutrient limitation of producer growth:

In [ ]:
# Monod kinetics parameters for phytoplankton
phyto_uptake = NutrientUptake(
    group_name='Phytoplankton',
    maximum_uptake_rate=2.0,    # μmax (day^-1)
    half_saturation_n=5.0,      # KN (μmol/L)
    half_saturation_p=0.5,      # KP (μmol/L)
    half_saturation_si=2.0,     # KSi (μmol/L)
    preference_n=0.6,           # N preference if limited
    preference_p=0.3,
    preference_si=0.1
)

print('Monod uptake kinetics (Phytoplankton):')
print(f'  Max uptake rate: {phyto_uptake.maximum_uptake_rate} day^-1')
print(f'  Half-saturation (N): {phyto_uptake.half_saturation_n} μmol/L')

## Part 5: Monod Growth Response

In [ ]:
# Calculate Monod growth response
n_conc = np.linspace(0, 20, 100)  # Nitrogen concentration (μmol/L)
kn = phyto_uptake.half_saturation_n
umax = phyto_uptake.maximum_uptake_rate

# Monod equation: U = Umax * [N] / (KN + [N])
uptake_rate = umax * n_conc / (kn + n_conc)

plt.figure(figsize=(10, 6))
plt.plot(n_conc, uptake_rate, linewidth=2, label='Monod kinetics')
plt.axhline(y=umax/2, color='r', linestyle='--', alpha=0.5, label=f'Umax/2')
plt.axvline(x=kn, color='r', linestyle='--', alpha=0.5, label=f'KN={kn}')
plt.xlabel('Nitrogen concentration (μmol/L)')
plt.ylabel('Uptake rate (day^-1)')
plt.title('Monod Growth Response - Phytoplankton')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

## Part 6: Nutrient Cycle Container

Manage nutrient cycling for entire system:

In [ ]:
# Create nutrient cycle system
nutrient_cycle = NutrientCycle()

# Add groups
nutrient_cycle.add_content(cod_nutrients)
nutrient_cycle.add_content(phytoplankton_nutrients)

# Add excretion
nutrient_cycle.add_excretion(cod_excretion)

# Add uptake
nutrient_cycle.add_uptake(phyto_uptake)

print(f'Nutrient cycle system created')
print(f'  Content definitions: {nutrient_cycle.n_content()}')
print(f'  Excretion paths: {nutrient_cycle.n_excretion()}')
print(f'  Uptake functions: {nutrient_cycle.n_uptake()}')

## Part 7: Stoichiometry Calculations

In [ ]:
# Example: Calculate nutrient budget for fish consumption
consumed_biomass = 100.0  # t of Cod consumed

# Nutrient content of consumed Cod
n_consumed = consumed_biomass * (cod_nutrients.nitrogen_percent / 100.0)
p_consumed = consumed_biomass * (cod_nutrients.phosphorus_percent / 100.0)

# Nutrient excretion
assimilation = 0.7  # 70% assimilation efficiency
n_excretion = n_consumed * assimilation * cod_excretion.nitrogen_excretion_rate
p_excretion = p_consumed * assimilation * cod_excretion.phosphorus_excretion_rate

print(f'Nutrient budget - Consumption of {consumed_biomass}t Cod:')
print(f'  N intake: {n_consumed:.2f} t')
print(f'  P intake: {p_consumed:.2f} t')
print(f'  N excretion: {n_excretion:.3f} t')
print(f'  P excretion: {p_excretion:.3f} t')

## Part 8: Nutrient Budget Closure

In [ ]:
# Verify nutrient balance in system
inputs = {'N': 500.0, 'P': 50.0}  # nutrient inputs from rivers
outputs = {'N': 450.0, 'P': 48.0}  # nutrient exports
burial = {'N': 30.0, 'P': 2.0}  # sedimentation

balance_df = pd.DataFrame([
    {'Element': 'N', 'Input': inputs['N'], 'Output': outputs['N'], 'Burial': burial['N']},
    {'Element': 'P', 'Input': inputs['P'], 'Output': outputs['P'], 'Burial': burial['P']}
])

balance_df['Accumulation'] = balance_df['Input'] - balance_df['Output'] - balance_df['Burial']

print('Nutrient budget closure:')
print(balance_df.to_string(index=False))

## Part 9: DataFrame Export

In [ ]:
# Export nutrient cycle configuration
nutrient_df = nutrient_cycle.to_dataframe()
print('Nutrient cycle configuration:')
print(nutrient_df)

# Save to CSV
nutrient_df.to_csv('nutrient_cycle.csv', index=False)
print('\nSaved to nutrient_cycle.csv')

## Performance Tips

- Monod calculations are vectorized using NumPy
- Pre-compute kinetic parameters during initialization
- Cache nutrient content values for repeated lookups
- Use matrix operations for large systems